<a href="https://colab.research.google.com/github/MMujtabaX/agentic-ai-journey/blob/main/03_tool_calling_agent/tool_calling_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 · Tool-Calling Agent with LangChain

An agent decides **when to call tools**, reads their results, and decides what to do next. This one is a pun-loving weather assistant with:

- a **real weather tool** (Open-Meteo, free, no API key)
- a **context-aware tool** that reads the user's ID from runtime context
- **conversation memory** across turns (`InMemorySaver` + `thread_id`)
- a printed **trace** of every tool call

Model: a tool-calling model on **Groq** (Llama 3.3 70B or GPT-OSS), selected automatically from what your account can access.

> 🔑 Add `GROQ_API_KEY` to Colab Secrets.

In [1]:
!pip install -qU langchain langchain-groq langgraph requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.2/250.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import requests
from dataclasses import dataclass
from google.colab import userdata
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver

from groq import Groq

# Groq retires models over time, so choose the first tool-calling model your account can access
available = {m.id for m in Groq(api_key=userdata.get("GROQ_API_KEY")).models.list().data}
for MODEL in ["llama-3.3-70b-versatile", "openai/gpt-oss-120b", "openai/gpt-oss-20b", "qwen/qwen3-32b"]:
    if MODEL in available:
        break
print("✅ Using model:", MODEL)

model = ChatGroq(model=MODEL, temperature=0, api_key=userdata.get("GROQ_API_KEY"))

✅ Using model: openai/gpt-oss-120b


## 1. Tools
A tool is a Python function with a docstring. The docstring is what the model reads to decide when to use it.

In [3]:
@tool
def get_weather_for_location(city: str) -> str:
    """Get the current weather for a given city."""
    geo = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                       params={"name": city, "count": 1}, timeout=10).json()
    if not geo.get("results"):
        return f"Could not find a city called {city}."
    place = geo["results"][0]
    wx = requests.get("https://api.open-meteo.com/v1/forecast",
                      params={"latitude": place["latitude"], "longitude": place["longitude"],
                              "current": "temperature_2m,wind_speed_10m,relative_humidity_2m"},
                      timeout=10).json()["current"]
    return (f"{place['name']}, {place.get('country', '')}: {wx['temperature_2m']}°C, "
            f"humidity {wx['relative_humidity_2m']}%, wind {wx['wind_speed_10m']} km/h")

@dataclass
class Context:
    """Runtime context passed in with every request."""
    user_id: str

USER_CITIES = {"1": "Karachi", "2": "Lahore"}

@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """Look up the current user's city from their user ID."""
    return USER_CITIES.get(runtime.context.user_id, "Islamabad")

print(get_weather_for_location.invoke({"city": "Karachi"}))

Karachi, Pakistan: 31.2°C, humidity 62%, wind 18.9 km/h


## 2. Create the agent

In [4]:
SYSTEM_PROMPT = """You are an expert weather forecaster who loves puns.

You have two tools:
- get_weather_for_location: get the current weather for a specific city
- get_user_location: find out where the user is

If the user asks about the weather "outside" or "here", first call get_user_location,
then get the weather for that city. Keep answers to 2-3 sentences."""

agent = create_agent(
    model=model,
    tools=[get_weather_for_location, get_user_location],
    system_prompt=SYSTEM_PROMPT,
    context_schema=Context,
    checkpointer=InMemorySaver(),
)

def show_trace(result):
    """Print every tool call and tool result, then the final answer."""
    for m in result["messages"]:
        for tc in getattr(m, "tool_calls", []) or []:
            print(f"  🔧 Tool call  : {tc['name']}({tc['args']})")
        if m.type == "tool":
            print(f"  📦 Tool result: {m.content}")
    print(f"\n🤖 {result['messages'][-1].content}")

## 3. Run it
The agent has to chain **two** tools: find the user's city, then fetch its weather.

In [5]:
config = {"configurable": {"thread_id": "conversation-1"}}

result = agent.invoke({"messages": [{"role": "user", "content": "What's the weather like outside?"}]},
                      config=config, context=Context(user_id="1"))
show_trace(result)

  🔧 Tool call  : get_user_location({})
  📦 Tool result: Karachi
  🔧 Tool call  : get_weather_for_location({'city': 'Karachi'})
  📦 Tool result: Karachi, Pakistan: 31.2°C, humidity 62%, wind 18.9 km/h

🤖 Looks like Karachi’s serving up a sizzling 31.2 °C with a humid 62 %—perfect for a “hot” pun: it’s so warm, even the clouds are sweating! Stay cool, and maybe chase a breeze before you melt!


## 4. Memory: a follow-up in the same thread
Because of the checkpointer and `thread_id`, the agent remembers the previous turn.

In [6]:
result = agent.invoke({"messages": [{"role": "user", "content": "And how does that compare with Lahore?"}]},
                      config=config, context=Context(user_id="1"))
show_trace(result)

  🔧 Tool call  : get_user_location({})
  📦 Tool result: Karachi
  🔧 Tool call  : get_weather_for_location({'city': 'Karachi'})
  📦 Tool result: Karachi, Pakistan: 31.2°C, humidity 62%, wind 18.9 km/h
  🔧 Tool call  : get_weather_for_location({'city': 'Lahore'})
  📦 Tool result: Lahore, Pakistan: 34.0°C, humidity 46%, wind 3.6 km/h

🤖 Lahore’s a touch hotter at 34 °C but drier (46 % humidity) and with a gentle 3.6 km/h breeze—so while Karachi’s feeling “steamy,” Lahore’s more of a “dry‑heat” roast. Both cities are definitely cooking up some serious summer vibes!


In [7]:
# A different user (different context) gets a different location
result = agent.invoke({"messages": [{"role": "user", "content": "Is it windy where I am?"}]},
                      config={"configurable": {"thread_id": "conversation-2"}}, context=Context(user_id="2"))
show_trace(result)

  🔧 Tool call  : get_user_location({})
  📦 Tool result: Lahore
  🔧 Tool call  : get_weather_for_location({'city': 'Lahore'})
  📦 Tool result: Lahore, Pakistan: 34.0°C, humidity 46%, wind 3.6 km/h

🤖 In Lahore it’s a gentle breeze at about 3.6 km/h—just enough to keep your hair in place, but not enough to fly a kite! 🌬️
